In [17]:
import pandas as pd
import numpy as np
import pickle

"""

countsdf will give summary about how many variants per kb are occurring in region with actiuvation domain vs non-activation domain

effect with high: high consequences of mutation (PTC), frame shift mutations
effect with moderate: nonsynonymous mutation
effect with low: sysnonynmous muattions

Full documentations: https://pcingola.github.io/SnpEff/snpeff/outputsummary/

"""

In [20]:
##less stringent threshold

variantdata=pd.read_csv('../maizedata/annotated_maize_TFs_variant_effects.csv')
TFs=pd.read_csv('../maizedata/maize_ActivityAnnotated.csv')

TFs['protein_ID']=TFs['protein_ID'].str.replace('_P', '_T0')
TFs['trancript']=[f"{x}_{y.split('_')[1]}" for x,y in zip(TFs['gene_ID'], TFs['protein_ID'])]
activators=set(TFs[TFs['Action']=='Activator']['trancript'])
variantdata.rename(columns={'gene': 'gene_ID'}, inplace=True)

file_path = '../maizedata/zma_all_preds.pkl' # Replace with your file's actual path
with open(file_path, 'rb') as file:
    data = pickle.load(file)
    data=data[['gene_ID','protein_ID','family','seq', 'activity_avg']]
data['protein_ID']=data['protein_ID'].str.replace('_P', '_T0')
data['trancript']=[f"{x}_{y.split('_')[1]}" for x,y in zip(data['gene_ID'], data['protein_ID'])]

##Less stringent threhsold

transcript_nomatch=set()

transcripts_activator=set()
for _, row in variantdata.iterrows():
    
    transcript=row['transcript']

    if not transcript in activators: continue

    arr=data.loc[data['trancript'] == transcript, 'activity_avg'].iloc[0]

    variantpos=int(row['protein_position'])-1

    if (len(arr)!=int(row['protein_length'])):
        # variantdata.at[_, 'activity']='None'
        transcript_nomatch.add(transcript)
        continue
    if int(row['protein_position'])>len(arr):
        print(transcript, int(row['protein_position']), len(arr)) ##because of stop codon not included
        continue

    transcripts_activator.add(transcript)

    if arr[variantpos]>1:
        variantdata.at[_, 'activity']='AD'
    else:
        variantdata.at[_, 'activity']='non-AD'
        
variantdata = variantdata[variantdata['activity'].isin(['AD', 'non-AD'])]

activatorlen=0
nonactivatorlen=0

for ta in transcripts_activator:
    arr=data.loc[data['trancript'] == ta, 'activity_avg'].iloc[0]

    for arr_num in arr:
        if arr_num>1:
            activatorlen+=1
        else:
            nonactivatorlen+=1

print(activatorlen, nonactivatorlen)

# counts = variantdata.groupby(['activity', 'effect']).size()
# # countsdf=pd.DataFrame(counts)

# # countsdf.columns=['effect', 'total']
countsdf = (
    variantdata
      .groupby(['activity', 'effect'])
      .size()
      .reset_index(name='total')
)
countsdf['length'] = np.where(
    countsdf['activity'] == 'AD',
    activatorlen,
    nonactivatorlen
)

countsdf['variantsperkb']=(countsdf['total']/countsdf['length'])*1000
countsdf

## we can see below that there is no bias for any kind of variants occurring if they exist in AD or no AD

Zm00001eb058820_T001 186 185
Zm00001eb075640_T001 293 292
Zm00001eb139470_T001 493 492
Zm00001eb153100_T001 1578 1577
Zm00001eb182300_T001 194 193
Zm00001eb187310_T001 608 607
Zm00001eb189840_T001 151 150
Zm00001eb201930_T001 290 289
Zm00001eb221620_T001 645 644
Zm00001eb288420_T001 453 452
Zm00001eb296640_T002 320 319
Zm00001eb313850_T001 419 418
Zm00001eb330710_T001 339 338
Zm00001eb344810_T001 348 347
Zm00001eb367200_T001 236 235
Zm00001eb425660_T001 946 945
25034 164382


,activity,effect,total,length,perkb
0,AD,HIGH,45,25034,1.797555
1,AD,LOW,886,25034,35.391867
2,AD,MODERATE,1022,25034,40.824479
3,non-AD,HIGH,285,164382,1.733766
4,non-AD,LOW,5854,164382,35.612172
5,non-AD,MODERATE,5947,164382,36.177927


In [21]:
# countsdf

In [23]:
## more stringent threshold

variantdata=pd.read_csv('../maizedata/annotated_maize_TFs_variant_effects.csv')
TFs=pd.read_csv('../maizedata/maize_ActivityAnnotated.csv')

TFs['protein_ID']=TFs['protein_ID'].str.replace('_P', '_T0')
TFs['trancript']=[f"{x}_{y.split('_')[1]}" for x,y in zip(TFs['gene_ID'], TFs['protein_ID'])]
activators=set(TFs[TFs['Action']=='Activator']['trancript'])
variantdata.rename(columns={'gene': 'gene_ID'}, inplace=True)

file_path = '../maizedata/zma_all_preds.pkl' # Replace with your file's actual path
with open(file_path, 'rb') as file:
    data = pickle.load(file)
data=data[['gene_ID','protein_ID','family','seq', 'activity_avg']]
data['protein_ID']=data['protein_ID'].str.replace('_P', '_T0')
data['trancript']=[f"{x}_{y.split('_')[1]}" for x,y in zip(data['gene_ID'], data['protein_ID'])]


LOW=-1
HIGH=0.5

transcript_nomatch=set()

transcripts_activator=set()
for _, row in variantdata.iterrows():
    
    transcript=row['transcript']

    if not transcript in activators: continue

    arr=data.loc[data['trancript'] == transcript, 'activity_avg'].iloc[0]

    variantpos=int(row['protein_position'])-1

    if (len(arr)!=int(row['protein_length'])):
        # variantdata.at[_, 'activity']='None'
        transcript_nomatch.add(transcript)
        continue
    if int(row['protein_position'])>len(arr):
        print(transcript, int(row['protein_position']), len(arr))
        continue

    transcripts_activator.add(transcript)

    if arr[variantpos]>3:
        variantdata.at[_, 'activity']='AD'
    elif (arr[variantpos]>=LOW) & (arr[variantpos]<=HIGH):
        variantdata.at[_, 'activity']='non-AD'
    # else:
    #     variantdata.at[_, 'activity']='non-AD'
variantdata = variantdata[variantdata['activity'].isin(['AD', 'non-AD'])]
    
activatorlen=0
nonactivatorlen=0

for ta in transcripts_activator:
    arr=data.loc[data['trancript'] == ta, 'activity_avg'].iloc[0]

    for arr_num in arr:
        if arr_num>3:
            activatorlen+=1
        elif (arr_num >= LOW) & (arr_num <= HIGH):
            nonactivatorlen+=1

print(activatorlen, nonactivatorlen)

countsdf = (
    variantdata
      .groupby(['activity', 'effect'])
      .size()
      .reset_index(name='total')
)
countsdf['length'] = np.where(
    countsdf['activity'] == 'AD',
    activatorlen,
    nonactivatorlen
)

countsdf['variantsperkb']=(countsdf['total']/countsdf['length'])*1000
countsdf

##Here as well, we don't see any bias for variants per kb occurring in AD vs non-AD regions for activators.

Zm00001eb058820_T001 186 185
Zm00001eb075640_T001 293 292
Zm00001eb139470_T001 493 492
Zm00001eb153100_T001 1578 1577
Zm00001eb182300_T001 194 193
Zm00001eb187310_T001 608 607
Zm00001eb189840_T001 151 150
Zm00001eb201930_T001 290 289
Zm00001eb221620_T001 645 644
Zm00001eb288420_T001 453 452
Zm00001eb296640_T002 320 319
Zm00001eb313850_T001 419 418
Zm00001eb330710_T001 339 338
Zm00001eb344810_T001 348 347
Zm00001eb367200_T001 236 235
Zm00001eb425660_T001 946 945
6911 140648


,activity,effect,total,length,perkb
0,AD,HIGH,13,6911,1.881059
1,AD,LOW,251,6911,36.318912
2,AD,MODERATE,267,6911,38.634062
3,non-AD,HIGH,230,140648,1.635288
4,non-AD,LOW,5041,140648,35.841249
5,non-AD,MODERATE,4999,140648,35.542631
